# Model Experiment — Logistic Regression (L1)

Sections:
1. Setup
2. Data Loading
3. Cleaning — MLflow runs per imputation strategy
4. Feature Engineering — engineered features ablation
5. Feature Selection — three selectors compared
6. Training and Hyperparameter Tuning
7. Cross-Validation of the best configuration
8. Final Pipeline + optional Model Registry registration

## 1. Setup

In [ ]:
import sys, os, warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('mlflow').setLevel(logging.ERROR)

for p in ['.', '..', '/kaggle/working/ML_Asgn2', '/kaggle/working']:
    if os.path.isdir(os.path.join(p, 'src')) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split

from src.data_utils import load_train, split_columns
from src.preprocessing import (
    build_linear_preprocessor, build_tree_preprocessor,
    CorrelationPruner, engineer_features,
)
from src.mlflow_utils import (
    init_tracking, named_run,
    evaluate_classifier, evaluate_train_val,
    cache_architecture_result,
)

RANDOM_STATE = 42
REGISTER_AS_BEST = False    # only flip to True in the winning architecture
SAMPLE_FRAC = 0.3           # set to None for the full dataset

In [ ]:
init_tracking('LogisticRegression_L1_Training')

## 2. Data Loading

In [ ]:
X, y = load_train(sample_frac=SAMPLE_FRAC, random_state=RANDOM_STATE)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

engineer = FunctionTransformer(engineer_features, validate=False)

_eng_sample = engineer.transform(X_train.head(200))
num_cols, cat_cols = split_columns(_eng_sample)
print(f'numeric cols:     {len(num_cols)}')
print(f'categorical cols: {len(cat_cols)}')
print(f'train rows: {len(X_train):,}  val rows: {len(X_val):,}')
print(f'fraud rate (train): {y_train.mean():.4f}')

In [ ]:
MODEL_TAG = 'LogReg_L1'

## 3. Cleaning

Linear models need scale-aware preprocessing:

- numeric: median-impute, then `StandardScaler`
- categorical: `'missing'` constant + `OneHotEncoder` with `min_frequency=0.001` and bounded `max_categories`

Several columns have thousands of unique values, so the OHE cap is essential. We probe two values of `max_categories` and keep the better one.

In [ ]:
from sklearn.linear_model import LogisticRegression

_probe_factory = lambda: LogisticRegression(penalty='l1', solver='liblinear', C=1.0, max_iter=200, random_state=RANDOM_STATE)

cleaning_results = {}
for max_cats in [10, 20]:
    pre = build_linear_preprocessor(num_cols, cat_cols, max_categories=max_cats)
    probe = Pipeline([('eng', engineer), ('pre', pre), ('clf', _probe_factory())])
    probe.fit(X_train, y_train)
    m = evaluate_train_val(probe, X_train, y_train, X_val, y_val)
    cleaning_results[max_cats] = m
    with named_run(f'{MODEL_TAG}_Cleaning_maxcats{max_cats}', tags={'stage':'cleaning'}):
        mlflow.log_param('max_categories', max_cats)
        mlflow.log_param('imputer', 'median + constant_missing')
        mlflow.log_param('scaler', 'StandardScaler')
        mlflow.log_metrics(m)

best_max_cats = max(cleaning_results, key=lambda k: cleaning_results[k]['val_roc_auc'])
print(f'best max_categories = {best_max_cats} '
      f"(val ROC-AUC={cleaning_results[best_max_cats]['val_roc_auc']:.4f})")
preprocessor = build_linear_preprocessor(num_cols, cat_cols, max_categories=best_max_cats)

## 4. Feature Engineering

Two engineered features (defined in `src/preprocessing.engineer_features` so they're shared across architectures and survive a round-trip through MLflow registry):

- `TransactionAmt_log` — log-scale of the heavy-tailed amount.
- `P_emaildomain_suffix` / `R_emaildomain_suffix` — TLD of payer / recipient email.

We compare the engineered probe to the cleaning-only baseline as a single MLflow run.

In [ ]:
probe = Pipeline([('eng', engineer), ('pre', preprocessor), ('clf', _probe_factory())])
probe.fit(X_train, y_train)
fe_metrics = evaluate_train_val(probe, X_train, y_train, X_val, y_val)
with named_run(f'{MODEL_TAG}_FeatureEngineering', tags={'stage':'feature_engineering'}):
    mlflow.log_param('engineered', 'TransactionAmt_log + email TLDs')
    mlflow.log_metrics(fe_metrics)
lift = fe_metrics['val_roc_auc'] - cleaning_results[best_max_cats]['val_roc_auc']
print(f"engineered val ROC-AUC = {fe_metrics['val_roc_auc']:.4f} (lift: {lift:+.4f})")

## 5. Feature Selection

Three approaches, each as its own MLflow run with `n_features_kept` logged:

- VarianceThreshold — drop the constant OHE columns the encoder produced.
- L1-LogReg embedded selection — `SelectFromModel(LogisticRegression(penalty='l1'))`.
- Tree-importance — small RF, useful counterpoint to the linear-only methods.

In [ ]:
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

selectors = {
    'variance': VarianceThreshold(threshold=0.001),
    'l1_logreg': SelectFromModel(
        LogisticRegression(penalty='l1', solver='liblinear', C=0.1,
                           max_iter=200, random_state=RANDOM_STATE),
        threshold='median',
    ),
    'rf_importance': SelectFromModel(
        RandomForestClassifier(n_estimators=60, max_depth=8,
                               n_jobs=-1, random_state=RANDOM_STATE),
        threshold='median',
    ),
}
def _selection_score(metrics):
    return metrics['val_roc_auc'] - 0.5 * max(0.0, metrics['overfit_gap'] - 0.02)

selection_results = {}
for name, sel in selectors.items():
    pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                     ('sel', sel), ('clf', _probe_factory())])
    pipe.fit(X_train, y_train)
    m = evaluate_train_val(pipe, X_train, y_train, X_val, y_val)
    m['selection_score'] = _selection_score(m)
    n_kept = int(np.asarray(sel.get_support()).sum())
    selection_results[name] = (sel, m, n_kept)
    with named_run(f'{MODEL_TAG}_FeatureSelection_{name}', tags={'stage':'feature_selection'}):
        mlflow.log_param('selector', name)
        mlflow.log_metric('n_features_kept', n_kept)
        loggable = {k: v for k, v in m.items() if isinstance(v, (int, float))}
        mlflow.log_metrics(loggable)

best_sel_name = max(selection_results, key=lambda k: selection_results[k][1]['selection_score'])
best_sel, best_sel_metrics, best_sel_kept = selection_results[best_sel_name]
print(f'best selector (by selection_score): {best_sel_name} '
      f'(val={best_sel_metrics["val_roc_auc"]:.4f}, '
      f'gap={best_sel_metrics["overfit_gap"]:.4f}, '
      f'kept={best_sel_kept})')

## 6. Training and Hyperparameter Tuning

We sweep a hand-picked grid that intentionally spans underfit / well-fit / overfit. Each combo is logged as its own MLflow run, named with its parameter values (`{TAG}_HP_<abbreviated_params>`).

Selection rule:

    selection_score = val_roc_auc - 0.5 * max(0, overfit_gap - 0.02)

This penalises configurations that buy marginal val AUC by overfitting. The chosen combo is then cross-validated in §7.

In [ ]:
from sklearn.linear_model import LogisticRegression

hp_grid = [
    {'C': 0.001},
    {'C': 0.01},
    {'C': 0.1},
    {'C': 1.0},
    {'C': 10.0},
]

def _short_run_name(tag, params):
    abbrev = {
        'n_estimators': 'n', 'learning_rate': 'lr', 'max_depth': 'd',
        'min_samples_leaf': 'msl', 'subsample': 'ss', 'colsample_bytree': 'cs',
        'C': 'C', 'max_iter': 'iter', 'base_depth': 'bd', 'min_samples_split': 'mss',
        'max_leaf_nodes': 'mln', 'reg_alpha': 'ra', 'reg_lambda': 'rl',
    }
    parts = []
    for k, v in params.items():
        key = abbrev.get(k, k)
        val = str(v).replace('.', 'p').replace('None', 'NA')
        parts.append(f'{key}{val}')
    return f'{tag}_HP_' + '_'.join(parts)

make_estimator = lambda params: LogisticRegression(
    **params, penalty='l1', solver='liblinear',
    max_iter=300, random_state=RANDOM_STATE)

results = []
for params in hp_grid:
    estimator = make_estimator(params)
    pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                     ('sel', best_sel), ('clf', estimator)])
    pipe.fit(X_train, y_train)
    m = evaluate_train_val(pipe, X_train, y_train, X_val, y_val)
    m['selection_score'] = _selection_score(m)
    m['params'] = params
    results.append(m)
    with named_run(_short_run_name(MODEL_TAG, params), tags={'stage':'hp_tuning'}):
        mlflow.log_params(params)
        loggable = {k: v for k, v in m.items() if k != 'params' and isinstance(v, (int, float))}
        mlflow.log_metrics(loggable)

results_df = (pd.DataFrame([{**r['params'],
                             'train_auc': round(r['train_roc_auc'], 4),
                             'val_auc':   round(r['val_roc_auc'],   4),
                             'gap':       round(r['overfit_gap'],   4),
                             'score':     round(r['selection_score'], 4)}
                            for r in results])
              .sort_values('score', ascending=False))
print(results_df.to_string(index=False))

best_idx = max(range(len(results)), key=lambda i: results[i]['selection_score'])
best_params = results[best_idx]['params']
best_train_val = results[best_idx]
print(f'\nBEST (by selection_score): {best_params}')
print(f'  val AUC = {best_train_val["val_roc_auc"]:.4f}, '
      f'gap = {best_train_val["overfit_gap"]:.4f}, '
      f'score = {best_train_val["selection_score"]:.4f}')

## 7. Cross-Validation of the best configuration

3-fold StratifiedKFold on the chosen HP combo. Stratification is required given the ~3.5% fraud rate. Mean ± std across folds is what we report in the README table.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

best_estimator = make_estimator(best_params)
best_pipe = Pipeline([('eng', engineer), ('pre', preprocessor),
                      ('sel', best_sel), ('clf', best_estimator)])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_validate(best_pipe, X_train, y_train, cv=cv,
                            scoring=['roc_auc', 'average_precision'],
                            return_train_score=True, n_jobs=1)

cv_summary = {
    'cv_train_roc_auc_mean': float(np.mean(cv_scores['train_roc_auc'])),
    'cv_val_roc_auc_mean':   float(np.mean(cv_scores['test_roc_auc'])),
    'cv_val_roc_auc_std':    float(np.std(cv_scores['test_roc_auc'])),
    'cv_val_pr_auc_mean':    float(np.mean(cv_scores['test_average_precision'])),
    'cv_overfit_gap':        float(np.mean(cv_scores['train_roc_auc'])
                                  - np.mean(cv_scores['test_roc_auc'])),
}
with named_run(f'{MODEL_TAG}_CrossValidation', tags={'stage':'cross_validation'}):
    mlflow.log_params(best_params)
    mlflow.log_metrics(cv_summary)
for k, v in cv_summary.items():
    print(f'  {k}: {v:.4f}')

## 8. Final Pipeline + optional Registration

Refit on `train + val` (val has done its job during HP tuning; deployed model gets all the data). Then:

1. Log the pipeline as an MLflow artifact with `signature` + `input_example`, so it can be loaded later and called on a raw DataFrame.
2. Cache the architecture's headline metrics to `results_cache.json` for the README table.
3. If `REGISTER_AS_BEST = True`, register the pipeline as `IEEEFraudBestModel`. Only flip this flag in the architecture you want as the production model.

In [ ]:
from mlflow.models.signature import infer_signature

X_full = pd.concat([X_train, X_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)
best_pipe.fit(X_full, y_full)

final_val = evaluate_classifier(best_pipe, X_val, y_val, prefix='final_val')

signature = infer_signature(X_train.head(5), best_pipe.predict_proba(X_train.head(5)))
with named_run(f'{MODEL_TAG}_FinalPipeline', tags={'stage':'final_pipeline'}) as final_run:
    mlflow.log_params(best_params)
    mlflow.log_metrics(final_val)
    mlflow.log_metrics(cv_summary)
    mlflow.sklearn.log_model(
        sk_model=best_pipe,
        name='pipeline',
        signature=signature,
        input_example=X_train.head(2),
    )
    final_run_id = final_run.info.run_id
    print(f'Logged final pipeline. run_id = {final_run_id}')
    print(f'  final-val ROC-AUC: {final_val["final_val_roc_auc"]:.4f}')
    print(f'  CV-val ROC-AUC: {cv_summary["cv_val_roc_auc_mean"]:.4f} '
          f'± {cv_summary["cv_val_roc_auc_std"]:.4f}')

cache_architecture_result('LogisticRegression_L1', {
    'best_params':          {k: (v if v is not None else 'None') for k, v in best_params.items()},
    'val_roc_auc':          float(best_train_val['val_roc_auc']),
    'val_pr_auc':           float(best_train_val['val_pr_auc']),
    'overfit_gap':          float(best_train_val['overfit_gap']),
    'cv_val_roc_auc_mean':  cv_summary['cv_val_roc_auc_mean'],
    'cv_val_roc_auc_std':   cv_summary['cv_val_roc_auc_std'],
    'cv_val_pr_auc_mean':   cv_summary['cv_val_pr_auc_mean'],
    'best_selector':        best_sel_name,
    'n_features_kept':      int(best_sel_kept),
    'final_run_id':         final_run_id,
})
print('cached to results_cache.json')

if REGISTER_AS_BEST:
    mv = mlflow.register_model(
        model_uri=f'runs:/{final_run_id}/pipeline',
        name='IEEEFraudBestModel',
    )
    print(f'  registered as IEEEFraudBestModel version {mv.version}')
else:
    print('  REGISTER_AS_BEST is False — skipping registration.')